# Tests: `fasterai.prune.pruner` (source `nbs/prune/pruner.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.core.criteria import large_final
from fasterai.prune.pruner import *

In [ ]:
from fastcore.test import *
import warnings

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

x = torch.randn(1, 3, 8, 8)

# A fraction means what it says: 0.4 removes ~40% of the filters
model = _test_model()
pruner = Pruner(model, 0.4, 'local', large_final, example_inputs=x)
params_before = sum(p.numel() for p in model.parameters())
pruner.prune_model()
test_close(model[0].out_channels / 16, 0.6, eps=0.05)
test_close(model[3].out_channels / 32, 0.6, eps=0.05)
assert sum(p.numel() for p in model.parameters()) < params_before

# Model still produces valid output after pruning
out = model(x)
test_eq(out.shape[0], 1)
test_eq(out.shape[1], 10)

# The ratio is stored as the fraction it was given, and handed to torch-pruning unscaled
test_eq(pruner.pruning_ratio, 0.4)
test_eq(pruner.default_pruning_ratio, 0.4)

# A percent is read as x/100 for one release, warns, and prunes exactly like the fraction
_pm = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _pp = Pruner(_pm, 40, 'local', large_final, example_inputs=x)
_pp.prune_model()
test_eq(_pp.pruning_ratio, 0.4)
test_eq(_pm[0].out_channels, model[0].out_channels)
test_eq({_x.category for _x in w}, {FutureWarning})

# Nothing to prune, out of range, or not a number
with ExceptionExpected(ValueError, regex='removes nothing'):
    Pruner(_test_model(), 0, 'local', large_final, example_inputs=x)
with ExceptionExpected(ValueError, regex='fraction'):
    Pruner(_test_model(), 150, 'local', large_final, example_inputs=x)
with ExceptionExpected(TypeError, regex='must be a number'):
    Pruner(_test_model(), '0.4', 'local', large_final, example_inputs=x)

# Per-layer dict: 0 leaves a layer alone
_dm = _test_model()
_dp = Pruner(_dm, {'0': 0.3, '3': 0}, 'local', large_final, example_inputs=x)
_dp.prune_model()
assert _dm[0].out_channels < 16, _dm[0].out_channels
test_eq(_dm[3].out_channels, 32)
test_eq(_dp.pruning_ratio, {'0': 0.3, '3': 0.0})

# An explicit 0 survives a non-zero default_pruning_ratio
_dm2 = _test_model()
_dp2 = Pruner(_dm2, {'0': 0.3, '3': 0}, 'local', large_final, example_inputs=x,
              default_pruning_ratio=0.5)
_dp2.prune_model()
test_eq(_dp2.default_pruning_ratio, 0.5)
assert _dm2[0].out_channels < 16, _dm2[0].out_channels
test_eq(_dm2[3].out_channels, 32)

# A bad per-layer value names its layer
with ExceptionExpected(ValueError, regex="'3'"):
    Pruner(_test_model(), {'0': 0.3, '3': 150}, 'local', large_final, example_inputs=x)

# A model whose parameters are all frozen traces nothing, so pruning would silently do nothing
_fm = _test_model()
_fm.requires_grad_(False)
with ExceptionExpected(ValueError, regex='requires_grad_'):
    Pruner(_fm, 0.4, 'local', large_final, example_inputs=x)

# ... and it prunes once the parameters require grad again
_fm.requires_grad_(True)
Pruner(_fm, 0.4, 'local', large_final, example_inputs=x).prune_model()
assert _fm[0].out_channels < 16, _fm[0].out_channels

# A tensor subclass (a fastai `TensorImage`, say) is traced as a plain tensor: without that,
# torch-pruning misses the batch-norm that follows a pruned convolution
class _MyTensor(torch.Tensor): pass

class _Residual(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1, self.bn1 = nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16)
        self.conv2, self.bn2 = nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16)
        self.pool, self.fc = nn.AdaptiveAvgPool2d(1), nn.Linear(16, 10)
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(x)); y += x
        return self.fc(self.pool(torch.relu(y)).flatten(1))

_rm = _Residual()
Pruner(_rm, 0.4, 'local', large_final, example_inputs=x.as_subclass(_MyTensor)).prune_model()
assert _rm.conv2.out_channels < 16, _rm.conv2.out_channels
test_eq(_rm.bn2.running_mean.numel(), _rm.conv2.out_channels)
test_eq(_rm(x).shape, (1, 10))